# Task 6 - Run and Analyze the Judge

## 1. Sanity Check (5 products)

We ran the judge on rows [0, 2, 9, 11, 13] before the full run. Key observations:

| Row | Product | Judge Grounding | Human Grounding | Match? |
|-----|---------|----------------|-----------------|--------|
| 0 | iPhone 15 Pro | good | good | ✅ |
| 2 | Pixel 8 Pro | bad | ok | ❌ |
| 9 | Garmin Forerunner | bad | bad | ✅ |
| 11 | GoPro HERO12 | good | good | ✅ |
| 13 | Nintendo Switch | bad | good | ❌ |

The judge correctly catches fabricated ratings (Garmin 4.7/5) and correctly passes clean descriptions (iPhone, GoPro). However, it's stricter than human evaluation on borderline cases — it flags Pixel's 4.7/5 rating as `bad` (human said `ok`) and flags Nintendo for minor issues the human overlooked. The prompt was kept as-is since the judge's strictness is defensible per the rubric.

## 2. Full Run (all 50 products)

The judge was run on all 50 products using `meta-llama/Meta-Llama-3.1-8B-Instruct` with structured output (Pydantic schema). Results stored in `assignment_01_judged.xlsx`.

In [ ]:
import os, json, time
import pandas as pd
from pydantic import BaseModel
from typing import Literal
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("NEBIUS_API_KEY")
)

class CriterionResult(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]

class JudgeOutput(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult

JUDGE_SYSTEM_PROMPT = """
You are an expert product description evaluator. Your job is to evaluate a generated product description against a strict rubric.

You will receive:
1. The ORIGINAL PRODUCT DATA (name, attributes, material, warranty) — this is the ground truth.
2. The GENERATED DESCRIPTION — this is what you are evaluating.

Evaluate the description on exactly 5 criteria. For each criterion, first write a brief explanation of your reasoning, then give a verdict of \"good\", \"ok\", or \"bad\".

## Rubric Definitions

### Fluency
- good: All sentences are clear and easy to follow. No confusing phrasing. Ideas flow logically with no abrupt jumps.
- ok: Overall understandable, but 1-2 sentences feel slightly awkward or clunky.
- bad: Multiple sentences are confusing, incomplete, or very awkward.

### Grammar
- good: 0-1 minor grammar, spelling, or punctuation errors. Consistent tense.
- ok: 2-5 minor errors, but text remains clearly understandable.
- bad: More than 5 errors, or critical mistakes that make sentences hard to read.

### Tone
- good: Clearly friendly and positive, not exaggerated or spammy. Focuses on customer benefits.
- ok: Mostly neutral or slightly friendly, could be more engaging.
- bad: Tone feels wrong for a sales page. Overly clickbait or unrealistic promises.

### Length
- good: Total word count between 50 and 90 words (inclusive).
- ok: Total word count between 40-49 or 91-110 words (inclusive).
- bad: Fewer than 40 words or more than 110 words.

### Grounding
Compare EVERY factual claim against the original product data.
- good: All factual statements supported by given fields. No invented specs. Generic marketing phrases allowed.
- ok: One minor inferred detail that is plausible but not strictly given. No major fabricated facts.
- bad: Any fabricated factual detail, invented rating, or contradiction with input data.

## Important Notes
- Count words carefully for Length.
- For Grounding, be strict: any specific fact not in the original data is fabricated.
- Generic marketing phrases are NOT grounding violations.
""".strip()

JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [ ]:
def judge_description(product_data, generated_description):
    user_prompt = (
        f"## Original Product Data\n"
        f"- Name: {product_data['product_name']}\n"
        f"- Attributes: {product_data['Product_attribute_list']}\n"
        f"- Material: {product_data['material']}\n"
        f"- Warranty: {product_data['warranty']}\n\n"
        f"## Generated Description\n"
        f"{generated_description}"
    )
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "judge_output",
                "strict": True,
                "schema": JudgeOutput.model_json_schema()
            }
        },
        temperature=0.0,
        max_tokens=1000
    )
    raw = response.choices[0].message.content
    return JudgeOutput.model_validate_json(raw)


def compute_final_score(verdicts):
    """Apply Task 1 pass/fail rules."""
    if verdicts.get("grounding") == "bad":
        return "fail"
    if verdicts.get("length") == "bad":
        return "fail"
    if verdicts.get("fluency") == "bad":
        return "fail"
    if verdicts.get("grammar") == "bad":
        return "fail"
    all_vals = list(verdicts.values()) + ["good", "good"]  # latency + cost (programmatic)
    g = sum(1 for v in all_vals if v == "good")
    o = sum(1 for v in all_vals if v == "ok")
    b = sum(1 for v in all_vals if v == "bad")
    return "pass" if (g >= 4 and o <= 2 and b == 0) else "fail"

In [ ]:
df = pd.read_excel("assignment_01.xlsx")

all_results = []
for idx, row in df.iterrows():
    product_data = {
        "product_name": row["product_name"],
        "Product_attribute_list": row["Product_attribute_list"],
        "material": row["material"],
        "warranty": row["warranty"]
    }
    parsed = judge_description(product_data, row["generated_description"])
    verdicts = {
        "fluency": parsed.fluency.verdict,
        "grammar": parsed.grammar.verdict,
        "tone": parsed.tone.verdict,
        "length": parsed.length.verdict,
        "grounding": parsed.grounding.verdict,
    }
    final = compute_final_score(verdicts)
    all_results.append({
        "judge_fluency": parsed.fluency.verdict,
        "judge_grammar": parsed.grammar.verdict,
        "judge_tone": parsed.tone.verdict,
        "judge_length": parsed.length.verdict,
        "judge_grounding": parsed.grounding.verdict,
        "judge_final_score": final,
        "judge_fluency_expl": parsed.fluency.explanation,
        "judge_grammar_expl": parsed.grammar.explanation,
        "judge_tone_expl": parsed.tone.explanation,
        "judge_length_expl": parsed.length.explanation,
        "judge_grounding_expl": parsed.grounding.explanation,
    })
    print(f"[{idx+1}/50] {row['product_name'][:35]} → {final}")

judge_df = pd.DataFrame(all_results)
out_df = pd.concat([df, judge_df], axis=1)
out_df.to_excel("assignment_01_judged.xlsx", index=False)
print(f"\nSaved. Pass: {(judge_df['judge_final_score']=='pass').sum()}, Fail: {(judge_df['judge_final_score']=='fail').sum()}")

## 3. Compare to Human Evaluation

Agreement rates between human scores (Task 3) and the all-at-once judge, on the 15 manually evaluated rows:

| Criterion | Agreement | Notes |
|-----------|-----------|-------|
| Fluency | 15/15 (100%) | Perfect agreement — both consistently rate fluency as good |
| Grammar | 14/15 (93%) | 1 disagreement: judge rated Bose [4] grammar as ok (human: good) |
| Tone | 14/15 (93%) | 1 disagreement: judge rated ASUS [49] tone as good (human: ok) |
| Length | 13/15 (87%) | Judge miscounted words for Bose [4] and Fitbit [10], rating them bad when they're in range |
| Grounding | 5/15 (33%) | Major divergence — see analysis below |

### Grounding Divergence Analysis

The judge and human disagree on grounding in 10/15 cases. The pattern:

- **Judge stricter than human (2 cases):** Pixel [2] and Nintendo [13] — judge rates `bad` where human rated `ok`/`good`. The judge catches fabrications the human was lenient on.
- **Judge more lenient than human (8 cases):** Sony [3], MacBook [7], Fitbit [10], Xbox [15], Instant Pot [16], Stanley [22], ThermoBall [27], Samsung SSD [43] — judge rates `good` where human rated `ok`. The judge doesn't flag minor inferences (e.g., "keeps drinks cold for hours") that the human considered borderline.

**Why this happens:** Grounding is inherently subjective at the ok/good boundary. The human evaluator flagged plausible-but-not-stated inferences as `ok`, while the judge either missed them (rating `good`) or was stricter on outright fabrications (rating `bad`). The 8B model lacks the nuance to consistently distinguish between "minor inference" and "fully grounded" — it tends to be binary (good or bad) rather than using the ok middle ground.

## 4. Criterion-by-Criterion Judging

Instead of evaluating all 5 criteria in one call, we ran the judge separately for each criterion (one call per criterion per product) on the same 15 rows.

In [ ]:
class SingleCriterionResult(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]

CRITERION_PROMPTS = {
    "fluency": """You are an expert evaluator. Evaluate ONLY the FLUENCY of the following product description.\n\n## Fluency Rubric\n- good: All sentences are clear and easy to follow. No confusing phrasing. Ideas flow logically.\n- ok: Overall understandable, but 1-2 sentences feel slightly awkward or clunky.\n- bad: Multiple sentences are confusing, incomplete, or very awkward.\n\nProvide an explanation, then a verdict of \"good\", \"ok\", or \"bad\".""",
    "grammar": """You are an expert evaluator. Evaluate ONLY the GRAMMAR of the following product description.\n\n## Grammar Rubric\n- good: 0-1 minor grammar, spelling, or punctuation errors. Consistent tense.\n- ok: 2-5 minor errors, but text remains clearly understandable.\n- bad: More than 5 errors, or critical mistakes.\n\nProvide an explanation, then a verdict of \"good\", \"ok\", or \"bad\".""",
    "tone": """You are an expert evaluator. Evaluate ONLY the TONE of the following product description.\n\n## Tone Rubric\n- good: Clearly friendly and positive, not exaggerated or spammy. Focuses on customer benefits.\n- ok: Mostly neutral or slightly friendly, could be more engaging.\n- bad: Tone feels wrong for a sales page. Overly clickbait or unrealistic promises.\n\nProvide an explanation, then a verdict of \"good\", \"ok\", or \"bad\".""",
    "length": """You are an expert evaluator. Evaluate ONLY the LENGTH of the following product description.\n\n## Length Rubric\n- good: Total word count between 50 and 90 words (inclusive).\n- ok: Total word count between 40-49 or 91-110 words (inclusive).\n- bad: Fewer than 40 words or more than 110 words.\n\nCount the words carefully. Provide an explanation with the word count, then a verdict.""",
    "grounding": """You are an expert evaluator. Evaluate ONLY the GROUNDING of the following product description.\n\nCompare EVERY factual claim against the original product data.\n- good: All factual statements supported by given fields. Generic marketing phrases allowed.\n- ok: One minor inferred detail that is plausible but not strictly given. No major fabricated facts.\n- bad: Any fabricated factual detail, invented rating, or contradiction with input data.\n\nBe strict: any specific fact not in the original data is fabricated. Generic marketing phrases are NOT violations.\n\nList each factual claim and whether it's supported, then give a verdict.""",
}

def judge_single_criterion(criterion, product_data, generated_description):
    user_prompt = (
        f"## Original Product Data\n"
        f"- Name: {product_data['product_name']}\n"
        f"- Attributes: {product_data['Product_attribute_list']}\n"
        f"- Material: {product_data['material']}\n"
        f"- Warranty: {product_data['warranty']}\n\n"
        f"## Generated Description\n"
        f"{generated_description}"
    )
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": CRITERION_PROMPTS[criterion]},
            {"role": "user", "content": user_prompt}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "criterion_result",
                "strict": True,
                "schema": SingleCriterionResult.model_json_schema()
            }
        },
        temperature=0.0,
        max_tokens=500
    )
    raw = response.choices[0].message.content
    return SingleCriterionResult.model_validate_json(raw)

In [ ]:
eval_rows = [0, 2, 3, 4, 7, 10, 11, 13, 15, 16, 18, 22, 27, 43, 49]
criteria = ["fluency", "grammar", "tone", "length", "grounding"]

iso_results = {}
for idx in eval_rows:
    row = df.iloc[idx]
    product_data = {
        "product_name": row["product_name"],
        "Product_attribute_list": row["Product_attribute_list"],
        "material": row["material"],
        "warranty": row["warranty"]
    }
    iso_results[idx] = {}
    for c in criteria:
        r = judge_single_criterion(c, product_data, row["generated_description"])
        iso_results[idx][c] = r.verdict
    print(f"[{idx}] {row['product_name'][:35]} done")

# Save
rows_out = []
for idx in eval_rows:
    row_data = {"row_index": idx, "product_name": df.iloc[idx]["product_name"]}
    for c in criteria:
        row_data[f"isolated_{c}"] = iso_results[idx][c]
    rows_out.append(row_data)
pd.DataFrame(rows_out).to_csv("judge_per_criterion.csv", index=False)
print("Saved judge_per_criterion.csv")

### Isolated vs All-at-once vs Human Agreement

| Criterion | Human vs All-at-once | Human vs Isolated | Change |
|-----------|---------------------|-------------------|--------|
| Fluency | 100% | 80% | ↓ worse |
| Grammar | 93% | 27% | ↓ much worse |
| Tone | 93% | 87% | ↓ slightly worse |
| Length | 87% | 20% | ↓ much worse |
| Grounding | 33% | 40% | ↑ slightly better |

### Why Isolating Criteria Changed Results

Isolating criteria made most results **worse**, not better. This is surprising but explainable:

1. **Grammar became much stricter (27% agreement):** When evaluating grammar in isolation, the judge nitpicks minor stylistic choices (e.g., double spaces, em-dashes) that it overlooks when evaluating all criteria together. The all-at-once approach provides context that helps the model calibrate severity.

2. **Length became unreliable (20% agreement):** The isolated length judge frequently miscounts words. In the all-at-once mode, the model counts once and moves on. In isolation, with a prompt focused entirely on counting, the model paradoxically overthinks and miscounts more often — a known issue with small LLMs and arithmetic tasks.

3. **Grounding slightly improved (33% → 40%):** This is the one criterion where isolation helped. With the full prompt focused on grounding, the model has more "attention budget" to carefully compare each claim against the input data.

4. **Fluency dropped (100% → 80%):** In isolation, the judge becomes more critical of minor flow issues that it doesn't notice when also evaluating grammar, tone, etc.

**Conclusion:** For this 8B model, all-at-once judging produces better overall agreement with human evaluation. Isolation helps for complex criteria (grounding) but hurts for simpler ones where the model over-analyzes.

## 5. Analysis

### 5a. Trade-offs: Human Evaluation vs LLM-as-a-Judge

| Dimension | Human Evaluation | LLM-as-a-Judge |
|-----------|-----------------|----------------|
| **Cost** | Expensive — requires paid human time per product. At 2-3 min per product, 50 products = ~2.5 hours of work. | Very cheap — ~$0.002 per product with Llama-3.1-8B. 50 products costs ~$0.10 total. |
| **Scale** | Doesn't scale — evaluating 10,000 products would take weeks. | Scales easily — 10,000 products in a few hours, fully automated. |
| **Consistency** | Inconsistent — human fatigue, mood, and interpretation drift over time. Different humans may disagree. | Highly consistent — same input always produces same output (temperature=0). No fatigue. |
| **Accuracy** | Higher accuracy on nuanced criteria (grounding ok vs good). Humans understand context and intent better. | Lower accuracy on nuanced criteria (33% grounding agreement). Tends to be binary (good/bad) rather than using the ok middle ground. |
| **Speed** | Slow — minutes per product. | Fast — ~15 seconds per product. |
| **Adaptability** | Can handle edge cases and update rubric interpretation on the fly. | Rigid — follows the prompt literally. May miss context that a human would catch. |

### 5b. Recommendation for Production (thousands of descriptions daily)

For a production system generating thousands of descriptions daily, I would recommend a **hybrid approach**:

1. **LLM-as-a-judge for all descriptions** — automated first pass using the all-at-once judge. This catches clear failures (grounding=bad, length=bad) at scale with high consistency and low cost.

2. **Human review for borderline cases** — any description where the judge gives mixed signals (e.g., grounding=ok, or disagreement between criteria) gets flagged for human review. This targets human effort where it matters most.

3. **Periodic calibration** — regularly compare a sample of judge verdicts against human evaluation to detect drift and update the judge prompt if needed.

This approach gives you the scale and consistency of automated evaluation while preserving human accuracy for the cases that matter most. The cost is dominated by the human review of ~10-20% of descriptions rather than 100%.